# P2C-64 v3.1 (`WACA_G2_V31`: Slim 8-Dim Stage 2.5 + Long-Term Parity/5-Zone Stage 3 + `WIDE_SAFE_BND`) — Google Colab 実行ノートブック

Egaroucid 公開学習データ（約2,551万局面）および既存の v2.0 Phase-1 ベースライン（`raw_ntuple64.p2c`）を再利用し、**P2C-64 v3.1（`WACA_G2_V31`）** を学習・ビット完全一致監査・純スカラー実戦対局（20ms / 50ms / 100ms・`fail_hard` vs `wide_safe_bnd`・v3.1 vs v3.0・MPC $\sigma$）するためのワンクリックノートブックです。

### v3.1 の 4 大改良点
1. **超軽量 Stage 2.5（8次元・3 SWAR popcount・Tier-1再ロード 0 回）**：v3.0 比で Stage 2.5 レイテンシを約 `-38%` 短縮し、`3.5/2.1` 広窓や 50ms/100ms 深読み帯での NPS 低下を圧縮。
2. **長期大域 Stage 3（32次元・30ブロック偶奇分離 Key-14・4近傍空きパリティ＋全64パターン5ゾーン幾何分解）**：`ply >= 24` の 4 近傍連結空き領域パリティ（`singles` $O(1)$ 高速パス＋2段展開 Flood-Fill）と、10手読みでも線形 N-Tuple に吸収されない全64パターン5ゾーン構造（`c/e/inner/subdiag/outer10`）＋30ブロック（最大ズレ±1手）Key-14 4ch SCReLU を統合。
3. **`WIDE_SAFE_BND`（`--tt-guard 4.5`）**：ゲート発動マージン（`M_25/M_3`）と TT 保護マージン（`tt_guard`）を分離し、偽カットを封印しつつ深読み帯での TT 境界再利用効率を回復。
4. **QAT 劣化バグの完全修正**：`alpha * lr_mult` のステップ幅連動平滑化＋最良 Float チェックポイント再読込＆新規 AdamW リセットからの低LR QAT（`0.12x → 0.02x`）により、QAT での Val RMSE 劣化を根絶。

- **既存成果物を完全保護**：既存 `cache_bin/{train.bin, val.bin}`・`run_egaroucid_25m_v20/`・`run_egaroucid_25m_v30/` は読み取り再利用のみ。v3.1 の全成果物は新規ディレクトリ `run_egaroucid_25m_v31/` へ保存します。

## Cell 1: GPU/VRAM 自動判定 ＆ Googleドライブマウント

In [ ]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

RUN_NAME = "run_egaroucid_25m_v31"
V20_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v20"
V30_RUN_DIR = "/content/drive/MyDrive/p2c64_runs/run_egaroucid_25m_v30"
DRIVE_DATA_DIR = "/content/drive/MyDrive/p2c64_data"
DRIVE_CACHE_DIR = f"{DRIVE_DATA_DIR}/cache_bin"
DRIVE_WORK_DIR = f"/content/drive/MyDrive/p2c64_runs/{RUN_NAME}"
LOCAL_DATA_DIR = "/content/data"
REPO_DIR = "/content/p2c64-othello"

os.makedirs(DRIVE_WORK_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

VRAM_GB = 0.0
if torch.cuda.is_available():
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU     : {torch.cuda.get_device_name(0)} ({VRAM_GB:.1f} GiB VRAM)")
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
else:
    print("Running on CPU runtime (recommended for Cell 6 pure-scalar 20ms/50ms/100ms matches & MPC sigma)")

TRAIN_BATCH = 32768 if VRAM_GB >= 20.0 else 16384
MLP_HIDDEN = 16   # Default fast pure-scalar H=16
TRAIN_H24 = False # Set True to also train optional H=24 variant in Cell 4B
print(f"Config  : RUN_NAME={RUN_NAME}, TRAIN_BATCH={TRAIN_BATCH}, MLP_HIDDEN={MLP_HIDDEN}, TRAIN_H24={TRAIN_H24}")
!lscpu | grep -E "Model name|CPU\(s\):|L1d|L2|L3" | head -n 10

## Cell 2: リポジトリ `clone` / `pull` ＆ C++ 純スカラービルド・単体テスト・22カーネル 0-SIMD ASM監査

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/Taichi111009/p2c64-othello.git"
REPO_DIR="/content/p2c64-othello"

if [ -d "$REPO_DIR/.git" ]; then
    cd "$REPO_DIR" && git fetch origin && git pull --ff-only origin main
else
    git clone "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
make clean all test audit
ls -lh build/

## Cell 3: 既存キャッシュ `train.bin` / `val.bin` および凍結 Phase-1 `raw_ntuple64.p2c` の再利用準備

In [ ]:
import os
import shutil
import subprocess

cache_tr = f"{DRIVE_CACHE_DIR}/train.bin"
cache_va = f"{DRIVE_CACHE_DIR}/val.bin"
local_tr = f"{LOCAL_DATA_DIR}/train.bin"
local_va = f"{LOCAL_DATA_DIR}/val.bin"

for src, dst in [(cache_tr, local_tr), (cache_va, local_va)]:
    if os.path.exists(dst):
        print("[OK] Local SSD cache present:", dst)
    else:
        assert os.path.exists(src), f"Required binary cache not found on Drive: {src}"
        print(f"[Cache Hit] Copying {src} -> {dst} ...")
        shutil.copy2(src, dst)

# Reuse frozen Phase-1 baseline from run_egaroucid_25m_v20 (0-min Phase 1!)
v20_raw = f"{V20_RUN_DIR}/raw_ntuple64.p2c"
v31_raw = f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c"
local_raw = f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c"

if os.path.exists(v31_raw):
    print("[OK] Reusing raw_ntuple64.p2c in v3.1 work dir:", v31_raw)
elif os.path.exists(v20_raw):
    print("[Reuse v2.0 Phase-1] Copying frozen raw_ntuple64.p2c -> v3.1 work dir (0-min Phase 1!)...")
    shutil.copy2(v20_raw, v31_raw)
else:
    print("[Fallback] v2.0 raw_ntuple64.p2c not found; running Phase 1 once...")
    subprocess.run([
        "python3", f"{REPO_DIR}/python/train_p2c64.py",
        "--train-bin", local_tr,
        "--val-bin", local_va,
        "--work-dir", DRIVE_WORK_DIR,
        "--phases", "phase1",
        "--resume",
        "--batch-size", str(TRAIN_BATCH),
        "--p1-epochs", "6",
    ], check=True)

shutil.copy2(v31_raw, local_raw)
!ls -lh /content/data/

## Cell 4: WACA-G2 v3.1 本学習（`H=16`: Slim 8-Dim Stage 2.5 → 凍結 → 30-Block EO Key-14 + 32→16→1 MLP + 修正版 QAT）

In [ ]:
!python3 {REPO_DIR}/python/train_waca_g2_v31.py \
    --train-bin {LOCAL_DATA_DIR}/train.bin \
    --val-bin {LOCAL_DATA_DIR}/val.bin \
    --raw-p2c {LOCAL_DATA_DIR}/raw_ntuple64.p2c \
    --work-dir {DRIVE_WORK_DIR} \
    --out-name waca_g2_v31.p2c \
    --mlp-hidden {MLP_HIDDEN} \
    --batch-size {TRAIN_BATCH} \
    --s25-epochs 3 \
    --s25-lr 2e-2 \
    --s25-smooth 1e-3 \
    --s3-epochs 7 \
    --s3-qat-epochs 2 \
    --qat-lr-mult 0.12 \
    --qat-lr-min 0.02 \
    --s3-emb-lr 8e-3 \
    --s3-mlp-lr 4e-3 \
    --s3-ply-smooth 8e-4 \
    --s3-blk-smooth 4e-3 \
    --resume

print("\nSaved v3.1 artifacts on Google Drive:")
!ls -lh {DRIVE_WORK_DIR}/*.p2c {DRIVE_WORK_DIR}/metrics_v31.*

## Cell 4B: （オプション）`H=24` バリアント追加学習（`TRAIN_H24 = True` のときのみ実行・Stage 2.5 は自動再利用）

In [ ]:
import subprocess

if TRAIN_H24:
    subprocess.run([
        "python3", f"{REPO_DIR}/python/train_waca_g2_v31.py",
        "--train-bin", f"{LOCAL_DATA_DIR}/train.bin",
        "--val-bin", f"{LOCAL_DATA_DIR}/val.bin",
        "--raw-p2c", f"{LOCAL_DATA_DIR}/raw_ntuple64.p2c",
        "--work-dir", DRIVE_WORK_DIR,
        "--out-name", "waca_g2_v31_h24.p2c",
        "--mlp-hidden", "24",
        "--batch-size", str(TRAIN_BATCH),
        "--s25-epochs", "3",
        "--s3-epochs", "7",
        "--s3-qat-epochs", "2",
        "--qat-lr-mult", "0.12",
        "--qat-lr-min", "0.02",
        "--resume",
    ], check=True)
else:
    print("[Skip] TRAIN_H24 is False. Set TRAIN_H24 = True in Cell 1 if you want to also train waca_g2_v31_h24.p2c.")

## Cell 5: C++/Python 全3段ビット完全一致（`S1=0 S2.5=0 S3=0`）＆ 16対称（$D_4 \times \mathbb{Z}_2$）監査

In [ ]:
import os
import subprocess

models = [f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c", f"{DRIVE_WORK_DIR}/waca_g2_v31.p2c"]
h24_p = f"{DRIVE_WORK_DIR}/waca_g2_v31_h24.p2c"
if os.path.exists(h24_p):
    models.append(h24_p)
v30_p = f"{V30_RUN_DIR}/waca_g2.p2c"
if os.path.exists(v30_p):
    models.append(v30_p)

val_bin = f"{LOCAL_DATA_DIR}/val.bin" if os.path.exists(f"{LOCAL_DATA_DIR}/val.bin") else f"{DRIVE_CACHE_DIR}/val.bin"
subprocess.run([
    "python3", f"{REPO_DIR}/python/test_bit_exact_io.py",
    "--data", val_bin,
    "--num-pos", "1000",
    "--models", *models,
    "--bench-bridge",
], check=True)

## Cell 6: （CPUランタイム推奨）完全公平ベンチマーク（MPC $\sigma$ ・同一ノード・20ms / 50ms / 100ms ・ `fail_hard` vs `wide_safe_bnd` ・ v3.1 vs v3.0）

GPUを切断し CPU ランタイムへ切り替えた後、**Cell 1 → Cell 2 → Cell 6** を実行してください。すべて完全公平条件（`--no-mpc-pure-v64`、双方同一 Std-MPC・256 MiB TT・残り10空き完全読み）で実行し、`run_egaroucid_25m_v31/` へ CSV 保存します：
1. **MPC $\sigma$ 計測**（`d2→d6`, `d4→d8`, 2,000局面）：`RAW_NTUPLE_64` vs `WACA_G2 (v3.0)`（存在時） vs `WACA_G2_V31`
2. **同一ノード（`nodes=20000`, 1,000局, Fair MPC, `fail_hard 2.5/1.5`）**
3. **持ち時間 `20ms`（各1,000局, Fair MPC）**：`fail_hard 2.5/1.5`, `fail_hard 3.5/2.1`, `wide_safe_bnd 2.5/1.5 (tt_guard=4.5)`, `wide_safe_bnd 3.5/2.1 (tt_guard=4.5)`
4. **持ち時間 `50ms`（各600局, Fair MPC）**：`wide_safe_bnd 3.5/2.1 (tt_guard=4.5)`, `fail_hard 3.5/2.1`
5. **持ち時間 `100ms`（各500局, Fair MPC）**：`wide_safe_bnd 3.5/2.1 (tt_guard=4.5)`, `fail_hard 3.5/2.1`
6. **直接対決 `WACA_G2_V31 vs WACA_G2 (v3.0)`**（`20ms`, 1,000局、`V30_RUN_DIR/waca_g2.p2c` が存在する場合に自動実行）

In [ ]:
import os
import subprocess

MATCH_BIN = f"{REPO_DIR}/build/match_engine"
MPC_BIN = f"{REPO_DIR}/build/measure_mpc_sigma"
RAW_P2C = f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c"
if not os.path.exists(RAW_P2C):
    RAW_P2C = f"{V20_RUN_DIR}/raw_ntuple64.p2c"
V31_P2C = f"{DRIVE_WORK_DIR}/waca_g2_v31.p2c"
V30_P2C = f"{V30_RUN_DIR}/waca_g2.p2c"
CSV_V31 = f"{DRIVE_WORK_DIR}/match_results_v31.csv"
val_bin = f"{LOCAL_DATA_DIR}/val.bin" if os.path.exists(f"{LOCAL_DATA_DIR}/val.bin") else f"{DRIVE_CACHE_DIR}/val.bin"

assert os.path.exists(RAW_P2C), f"Missing RAW model: {RAW_P2C}"
assert os.path.exists(V31_P2C), f"Missing v3.1 model: {V31_P2C}"

# 1) MPC sigma measurement (d2->d6 and d4->d8, 2,000 pos)
mpc_models = [RAW_P2C]
if os.path.exists(V30_P2C):
    mpc_models.append(V30_P2C)
mpc_models.append(V31_P2C)

for sh, dp in [(2, 6), (4, 8)]:
    print(f"\n=== P2C-64 v3.1 MPC Sigma: depth {sh} -> {dp} (2,000 pos) ===")
    subprocess.run([
        MPC_BIN,
        "--data", val_bin,
        "--num-pos", "2000",
        "--shallow", str(sh),
        "--deep", str(dp),
        "--tt-mb", "64",
        "--gate-mode", "fail_hard",
        "--m25", "2.5",
        "--m3", "1.5",
        "--csv", f"{DRIVE_WORK_DIR}/mpc_sigma_v31_d{sh}_d{dp}.csv",
        "--models", *mpc_models,
    ], check=True)

# 2) Fair benchmark suite: WACA_G2_V31 vs RAW_NTUPLE_64
suite_v31 = [
    ("1. Equal-Nodes 20k Fair MPC (fail_hard 2.5/1.5)",               "nodes", "20000", "500", "fail_hard",     "2.5", "1.5", "4.5"),
    ("2. Time 20ms Fair MPC (fail_hard 2.5/1.5, 1000g)",              "time",  "20",    "500", "fail_hard",     "2.5", "1.5", "4.5"),
    ("3. Time 20ms Fair MPC (fail_hard 3.5/2.1, 1000g)",              "time",  "20",    "500", "fail_hard",     "3.5", "2.1", "4.5"),
    ("4. Time 20ms Fair MPC (wide_safe_bnd 2.5/1.5 g=4.5, 1000g)",    "time",  "20",    "500", "wide_safe_bnd", "2.5", "1.5", "4.5"),
    ("5. Time 20ms Fair MPC (wide_safe_bnd 3.5/2.1 g=4.5, 1000g)",    "time",  "20",    "500", "wide_safe_bnd", "3.5", "2.1", "4.5"),
    ("6. Time 50ms Fair MPC (wide_safe_bnd 3.5/2.1 g=4.5, 600g)",     "time",  "50",    "300", "wide_safe_bnd", "3.5", "2.1", "4.5"),
    ("7. Time 50ms Fair MPC (fail_hard 3.5/2.1, 600g)",               "time",  "50",    "300", "fail_hard",     "3.5", "2.1", "4.5"),
    ("8. Time 100ms Fair MPC (wide_safe_bnd 3.5/2.1 g=4.5, 500g)",    "time",  "100",   "250", "wide_safe_bnd", "3.5", "2.1", "4.5"),
    ("9. Time 100ms Fair MPC (fail_hard 3.5/2.1, 500g)",              "time",  "100",   "250", "fail_hard",     "3.5", "2.1", "4.5"),
]

for label, mode, limit, pairs, gmode, m25, m3, ttg in suite_v31:
    print(f"\n=== [v3.1 {label}] WACA_G2_V31 vs RAW_NTUPLE_64 ===")
    subprocess.run([
        MATCH_BIN,
        "--p1", V31_P2C,
        "--p2", RAW_P2C,
        "--mode", mode,
        "--limit", limit,
        "--pairs", pairs,
        "--opening-plies", "8",
        "--endgame-empties", "10",
        "--tt-mb", "256",
        "--mpc", "--mpc-t", "1.5",
        "--no-mpc-pure-v64",
        "--gate-mode", gmode,
        "--m25", m25,
        "--m3", m3,
        "--tt-guard", ttg,
        "--seed", "20261126",
        "--csv", CSV_V31,
    ], check=True)

# 3) Head-to-head v3.1 vs v3.0 (if v3.0 waca_g2.p2c exists on Drive)
if os.path.exists(V30_P2C):
    print("\n=== [v3.1 vs v3.0 Head-to-Head] Time 20ms Fair MPC (wide_safe_bnd 3.5/2.1 g=4.5, 1000g) ===")
    subprocess.run([
        MATCH_BIN,
        "--p1", V31_P2C,
        "--p2", V30_P2C,
        "--mode", "time",
        "--limit", "20",
        "--pairs", "500",
        "--opening-plies", "8",
        "--endgame-empties", "10",
        "--tt-mb", "256",
        "--mpc", "--mpc-t", "1.5",
        "--no-mpc-pure-v64",
        "--gate-mode", "wide_safe_bnd",
        "--m25", "3.5",
        "--m3", "2.1",
        "--tt-guard", "4.5",
        "--seed", "20261126",
        "--csv", CSV_V31,
    ], check=True)

print(f"\nAll v3.1 match & MPC results saved to: {DRIVE_WORK_DIR}")
!cat {CSV_V31}